In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models  #type:ignore
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error

In [2]:
cols = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

def load_data(path):
    return pd.read_csv(path, sep=' +', header=None,
                       usecols=range(26), names=cols, engine='python')

train = load_data('../Dataset/train_FD004.txt')
test  = load_data('../Dataset/test_FD004.txt')
rul   = pd.read_csv('../Dataset/RUL_FD004.txt', header=None, names=['rul'])

In [3]:
train.head()

,unit,cycle,os1,os2,os3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 61249 entries, 0 to 61248
Data columns (total 26 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   unit    61249 non-null  int64  
 1   cycle   61249 non-null  int64  
 2   os1     61249 non-null  float64
 3   os2     61249 non-null  float64
 4   os3     61249 non-null  float64
 5   s1      61249 non-null  float64
 6   s2      61249 non-null  float64
 7   s3      61249 non-null  float64
 8   s4      61249 non-null  float64
 9   s5      61249 non-null  float64
 10  s6      61249 non-null  float64
 11  s7      61249 non-null  float64
 12  s8      61249 non-null  float64
 13  s9      61249 non-null  float64
 14  s10     61249 non-null  float64
 15  s11     61249 non-null  float64
 16  s12     61249 non-null  float64
 17  s13     61249 non-null  float64
 18  s14     61249 non-null  float64
 19  s15     61249 non-null  float64
 20  s16     61249 non-null  float64
 21  s17     61249 non-null  int64  
 22  s18     6

In [5]:
DROP_SENSORS = ['s1','s5','s6','s8','s10','s13','s15','s16','s18','s19']
KEEP_SENSORS = ['s2','s3','s4','s7','s9','s11','s12','s14','s17','s20','s21']

train.drop(columns=DROP_SENSORS, inplace=True)
test.drop(columns=DROP_SENSORS, inplace=True)

In [6]:
def add_rul(df):
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    df['RUL'] = max_cycle - df['cycle']
    return df

train = add_rul(train)

last_cycles = test.groupby('unit')['cycle'].max().reset_index()
last_cycles['RUL'] = rul['rul'].values
test = test.merge(last_cycles[['unit','RUL']], on='unit', how='left')
test_last = test.groupby('unit').last().reset_index()

In [7]:
RUL_CLIP = 125
train['RUL'] = train['RUL'].clip(upper=RUL_CLIP)

In [8]:
SETTING_COLS = ['os1', 'os2', 'os3']
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
train['condition'] = kmeans.fit_predict(train[SETTING_COLS])
test['condition']  = kmeans.predict(test[SETTING_COLS])

In [9]:
train[KEEP_SENSORS] = train[KEEP_SENSORS].astype(np.float64)
test[KEEP_SENSORS]  = test[KEEP_SENSORS].astype(np.float64)

scalers = {}
for cond in range(6):
    scaler = MinMaxScaler()
    mask_tr = train['condition'] == cond
    mask_te = test['condition']  == cond
    train.loc[mask_tr, KEEP_SENSORS] = scaler.fit_transform(train.loc[mask_tr, KEEP_SENSORS])
    if mask_te.sum() > 0:
        test.loc[mask_te, KEEP_SENSORS] = scaler.transform(test.loc[mask_te, KEEP_SENSORS])
    scalers[cond] = scaler

In [10]:
print("=== RUL range (train should max at 125) ===")
print(f"Train RUL: min={train['RUL'].min()}, max={train['RUL'].max()}")
print(f"Test  RUL: min={test_last['RUL'].min()}, max={test_last['RUL'].max()}")

print("\n=== Sensor range (should be 0.0 to 1.0) ===")
print("Train:", train[KEEP_SENSORS].min().min().round(3), "to", train[KEEP_SENSORS].max().max().round(3))
print("Test: ", test[KEEP_SENSORS].min().min().round(3),  "to", test[KEEP_SENSORS].max().max().round(3))


=== RUL range (train should max at 125) ===
Train RUL: min=0, max=125
Test  RUL: min=6, max=195

=== Sensor range (should be 0.0 to 1.0) ===
Train: 0.0 to 1.0
Test:  -0.091 to 1.009


In [11]:
WINDOW = 30

def build_sequences(df, feature_cols, window=WINDOW):
    X, y = [], []
    for _, engine_df in df.groupby('unit'):
        engine_df = engine_df.sort_values('cycle')
        data, labels = engine_df[feature_cols].values, engine_df['RUL'].values
        for i in range(len(data) - window + 1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_last_sequences(df, feature_cols, window=WINDOW):
    X, y = [], []
    for _, engine_df in df.groupby('unit'):
        engine_df = engine_df.sort_values('cycle')
        data = engine_df[feature_cols].values
        if len(data) >= window:
            X.append(data[-window:])
        else:
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(engine_df['RUL'].iloc[-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [12]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train, groups=train['unit']))

X_train, y_train = build_sequences(train.iloc[train_idx], KEEP_SENSORS, WINDOW)
X_val,   y_val   = build_sequences(train.iloc[val_idx],   KEEP_SENSORS, WINDOW)
X_test,  y_test  = build_last_sequences(test, KEEP_SENSORS, WINDOW)

y_test_clipped = np.clip(y_test, 0, RUL_CLIP)

print(f"\n=== Sequence shapes ===")
print(f"X_train={X_train.shape}, y_train max={y_train.max():.1f} (should be ≤125)")
print(f"X_val={X_val.shape},   y_val max={y_val.max():.1f}")
print(f"X_test={X_test.shape},  y_test max={y_test.max():.1f}")


=== Sequence shapes ===
X_train=(43523, 30, 11), y_train max=125.0 (should be ≤125)
X_val=(10505, 30, 11),   y_val max=125.0
X_test=(248, 30, 11),  y_test max=195.0


In [13]:
def build_gru(window, n_features):
    inp = tf.keras.Input(shape=(window, n_features))
    x = layers.GRU(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(32)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='linear')(x)
    return models.Model(inp, out)

In [14]:
model = build_gru(WINDOW, len(KEEP_SENSORS))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss=tf.keras.losses.Huber(delta=9.0),  # not the string 'huber' — that locks delta=1.0
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30, 11)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 30, 128)        │        54,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 30, 64)         │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,889 (398.00 KB)

 Trainable params: 101,889 (398.00 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-5, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=512,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
86/86 ━━━━━━━━━━━━━━━━━━━━ 24s 223ms/step - loss: 727.4158 - rmse: 94.6929 - val_loss: 675.0862 - val_rmse: 89.2951 - learning_rate: 3.0000e-04
Epoch 2/50
29/86 ━━━━━━━━━━━━━━━━━━━━ 11s 200ms/step - loss: 685.0270 - rmse: 89.9978

KeyboardInterrupt: 

In [ ]:
def nasa_score(y_true, y_pred):
    d = y_pred.flatten() - y_true.flatten()
    return float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))

val_preds  = model.predict(X_val).flatten()
test_preds = np.clip(model.predict(X_test).flatten(), 0, RUL_CLIP)

print(f"Val RMSE:   {np.sqrt(mean_squared_error(y_val, val_preds)):.2f}")
print(f"Test RMSE:  {np.sqrt(mean_squared_error(y_test_clipped, test_preds)):.2f}")
print(f"NASA Score: {nasa_score(y_test_clipped, test_preds):.1f}")
print(f"Pred range: {test_preds.min():.1f} – {test_preds.max():.1f}")
print(f"True range: {y_test_clipped.min():.1f} – {y_test_clipped.max():.1f}")

In [ ]:
results = pd.DataFrame({
    'unit': test_last['unit'].values,
    'true_RUL': test_last['RUL'].values,
    'pred_RUL': test_preds.flatten()
})
results.head(10)
